In [1]:
%run -n "/Users/alejandrogomez-paz/Desktop/RAG/rag_code/old_RAG.ipynb"

Total chunks: 41
Example meta: {'source': 'OSHA — Radiation Emergency Preparedness and Response', 'page': 'Health and Safety Hazards during Radiation Emergencies', 'chunk_id': 'osha_rep_hazards'}


/opt/anaconda3/envs/py_env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 2/2 [00:00<00:00,  4.76it/s]


FAISS index size: 41
The threshold for ARS depends on the dose received and the rate of exposure.

- At approximately 200 rem, transient skin erythema may appear within 2-24 hours.
- At 9000 rem or greater (whole body acute), death can occur from cerebral/vascular breakdown.
- The LD50/30 — the whole body dose that results in lethality to 50% of an exposed population within 30 days — is estimated at 9000 rem.

Doses of 100 rad (1 Gy) or greater raise concern for acute death risk.


In [5]:
import json, math, re, time, statistics, os, sys
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/local_LLM")
sys.path.append(r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1")
from rad_ai import query
from benchmark_golden_pairs_1 import golden_pairs, should_refuse_map, answer_keys
from dotenv import load_dotenv
from groq import Groq

load_dotenv()
_judge_client = Groq(api_key=os.environ["GROQ_API_KEY"])


CORPUS_PATH = r"/Users/alejandrogomez-paz/Desktop/RAG/corpus_v1/corpus_1.jsonl"
REFUSAL = "don't have enough information"
DEFAULT_K = 5
N_RUNS = 3

QUALITY_WEIGHTS = {'AC': 0.30, 'CIF1': 0.20, 'REC': 0.15, 'COPG': 0.10, 'RR': 0.15, 'HR': 0.10}

metas = [
    {"source": r["source"], "page": r["section"], "chunk_id": r["chunk_id"]}
    for r in (json.loads(l) for l in open(CORPUS_PATH, encoding="utf-8"))
    if r.get("record_type") == "chunk"
]
cid2sp = {m["chunk_id"]: f'{m["source"]} | page {m["page"]}' for m in metas}
golden_sp = {q: {cid2sp[c] for c in ids} for q, ids in golden_pairs.items()}


def get_citation(hit):
    """Comparable source|page string for a retrieved hit."""
    return f'{hit["source"]} | page {hit["page"]}'

def context_precision(q, hits):
    """Fraction of retrieved chunks that are golden."""
    if not hits:
        return 0.0
    g = golden_sp.get(q, set())
    return sum(1 for h in hits if get_citation(h) in g) / len(hits)

def context_precision_at_g(q, hits):
    """Correct retrieved / min(k, #golden); removes the single-golden precision cap."""
    g = golden_sp.get(q, set())
    if not g:
        return None
    got = {get_citation(h) for h in hits}
    denom = min(len(hits), len(g))
    return len(got & g) / denom if denom else 0.0

def context_recall(q, hits):
    """Fraction of golden chunks that were retrieved (Hit@k)."""
    g = golden_sp.get(q, set())
    if not g:
        return None
    got = {get_citation(h) for h in hits}
    return len(got & g) / len(g)

def ndcg(q, hits):
    """Rank-aware retrieval quality over binary golden relevance."""
    g = golden_sp.get(q, set())
    if not g:
        return None
    dcg, seen = 0.0, set()
    for i, h in enumerate(hits):
        c = get_citation(h)
        if c in g and c not in seen:
            seen.add(c)
            dcg += 1 / math.log2(i + 2)
    idcg = sum(1 / math.log2(i + 2) for i in range(min(len(g), len(hits))))
    return dcg / idcg if idcg else 0.0

def citation_f1(cited, golden):
    """Harmonic mean of citation precision and citation recall."""
    if not golden:
        return None
    p = sum(1 for c in cited if c in golden) / len(cited) if cited else 0.0
    r = sum(1 for c in golden if c in set(cited)) / len(golden)
    return 2 * p * r / (p + r) if (p + r) else 0.0

def answer_correctness(answer, key):
    """Fraction of required phrases present, or token-F1 vs a gold answer string."""
    if key is None or refused(answer):
        return None if key is None else 0.0
    ans = (answer.get("answer", "") if isinstance(answer, dict) else str(answer)).lower()
    if isinstance(key, dict):
        key = key.get("must_include") or key.get("keywords") or key.get("answer")
    if isinstance(key, (list, tuple, set)):
        return sum(1 for p in key if str(p).lower() in ans) / len(key) if key else None
    gold, got = set(re.findall(r"[a-z0-9]+", str(key).lower())), set(re.findall(r"[a-z0-9]+", ans))
    o = len(gold & got)
    return 2 * o / (len(gold) + len(got)) if o else 0.0

def refused(answer):
    """True if the pipeline abstained (structured flag or refusal phrase)."""
    if isinstance(answer, dict) and "refused" in answer:
        return bool(answer["refused"])
    text = answer.get("answer", "") if isinstance(answer, dict) else str(answer)
    return REFUSAL in text.lower()

def refusal_score(did, should):
    """Fraction of correct refuse/answer decisions."""
    return sum(1 for d, s in zip(did, should) if d == s) / len(should)

def judge_hallucination(q, answer, chunks, retries=2):
    """Independent LLM judge (Groq, not the local generator).
    True = answer has a claim unsupported by the chunks, False = grounded,
    None = judge unreachable (excluded from HR rather than scored as clean)."""
    if refused(answer):
        return False
    ans = answer.get("answer", "") if isinstance(answer, dict) else str(answer)
    if not ans.strip():
        return None
    ctx = "\n\n".join(f"[{i+1}] {h['text']}" for i, h in enumerate(chunks)) or "(none)"

    system = (
        "You are a strict factuality judge for a retrieval-augmented QA system. "
        "Decide whether the ANSWER contains a hallucination: any factual claim not "
        "supported by the CONTEXT. Judge only against the CONTEXT, not outside knowledge. "
        "A refusal is NOT a hallucination. List unsupported claims first, then the verdict. "
        'Respond ONLY as JSON: {"unsupported": ["..."], "hallucinated": true|false}'
    )
    user = f"QUESTION:\n{q}\n\nCONTEXT:\n{ctx}\n\nANSWER:\n{ans}"

    for attempt in range(retries + 1):
        try:
            resp = _judge_client.chat.completions.create(
                model=JUDGE_MODEL, temperature=0,
                response_format={"type": "json_object"},
                messages=[{"role": "system", "content": system},
                          {"role": "user", "content": user}],
            )
            raw = resp.choices[0].message.content
            m = re.search(r"\{.*\}", raw, re.DOTALL)
            if m:
                return bool(json.loads(m.group(0)).get("hallucinated", False))
        except Exception:
            if attempt < retries:
                time.sleep(2 ** attempt)
    return None

def context_tokens(hits):
    """Estimated context tokens sent to the LLM (~4 chars/token)."""
    return sum(len(h["text"]) for h in hits) // 4

def quality_score(s):
    """Weighted composite of the quality metrics; HR inverted, weights normalized."""
    parts = {'AC': s['AC'], 'CIF1': s['CIF1'], 'REC': s['REC'],
             'COPG': s['COPG'], 'RR': s['RR'], 'HR': 1 - s['HR']}
    num = sum(QUALITY_WEIGHTS[k] * v for k, v in parts.items() if v is not None)
    wsum = sum(QUALITY_WEIGHTS[k] for k, v in parts.items() if v is not None)
    return num / wsum if wsum else 0.0

def _mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else 0.0

def run_once(k=DEFAULT_K):
    """One full pass over the eval set; returns (quality, scores dict)."""
    lat, cop, copg, rec, nd, cif, ac, tok = [], [], [], [], [], [], [], []
    did, should, hall = [], [], []
    for q, sr in should_refuse_map.items():
        t0 = time.time()
        hits = [{"source": m["source"], "page": m["page"], "chunk_id": m["chunk_id"], "text": txt}
                for _, txt, m in retrieve(q, k=k)]
        answer = rag_answer(q, k=k)
        lat.append(time.time() - t0)
        tok.append(context_tokens(hits))

        cop.append(context_precision(q, hits))
        for lst, fn in ((copg, context_precision_at_g), (rec, context_recall), (nd, ndcg)):
            v = fn(q, hits)
            if v is not None:
                lst.append(v)

        if not sr:
            ans = answer.get("answer", answer) if isinstance(answer, dict) else answer
            cited = [get_citation(h) for h in hits if h["source"].lower() in str(ans).lower()]
            f = citation_f1(cited, golden_sp.get(q, set()))
            if f is not None:
                cif.append(f)
            a = answer_correctness(answer, answer_keys.get(q))
            if a is not None:
                ac.append(a)
            hall.append(judge_hallucination(q, answer, hits))

        did.append(refused(answer))
        should.append(sr)

    scores = {'COP': _mean(cop), 'COPG': _mean(copg), 'REC': _mean(rec), 'NDCG': _mean(nd),
              'CIF1': _mean(cif), 'AC': _mean(ac), 'RR': refusal_score(did, should),
              'HR': _mean(hall), 'RT': _mean(lat), 'CTX_TOK': _mean(tok)}
    return quality_score(scores), scores

def benchmark(k=DEFAULT_K, n_runs=N_RUNS):
    """Run n_runs times; returns (quality, scores) each as mean ± 95% CI half-width."""
    def ci(xs):
        m = statistics.mean(xs)
        return m, (1.96 * statistics.stdev(xs) / math.sqrt(len(xs)) if len(xs) > 1 else 0.0)
    runs = [run_once(k) for _ in range(n_runs)]
    quals, scores = [q for q, _ in runs], [s for _, s in runs]
    return ci(quals), {m: ci([s[m] for s in scores]) for m in scores[0]}


In [6]:
import csv

DEFAULT_K = 3
N_RUNS = 1

METRIC_DOCS = {
    'COP':     "Context Precision: fraction of retrieved chunks that are golden.",
    'COPG':    "Context Precision @ golden-count: correct / min(k, #golden).",
    'REC':     "Context Recall (Hit@k): fraction of golden chunks retrieved.",
    'NDCG':    "Normalized DCG@k: rank-aware retrieval quality, golden nearer top scores higher.",
    'CIF1':    "Citation F1: harmonic mean of citation precision and recall.",
    'AC':      "Answer Correctness: required phrases present / token-F1 vs gold answer.",
    'RR':      "Refusal accuracy: fraction of correct refuse/answer decisions.",
    'HR':      "Hallucination Rate: fraction of answers unsupported by context (lower better).",
    'RT':      "Latency: mean retrieval + generation time per query (seconds).",
    'CTX_TOK': "Mean estimated context tokens sent to the LLM (~4 chars/token).",
}


def write_csv(path, quality, agg):
    """Write one row per metric (value, 95% CI, explanation) plus the composite quality.

    quality: (mean, ci) tuple for the composite score.
    agg:     {metric: (mean, ci)} as returned by benchmark().
    """
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["metric", "value", "ci95_halfwidth", "explanation"])
        for m, (v, c) in agg.items():
            w.writerow([m, round(v, 4), round(c, 4), METRIC_DOCS.get(m, "")])
        qv, qc = quality
        w.writerow(["quality", round(qv, 4), round(qc, 4),
                    "Weighted composite of AC, CIF1, REC, COPG, RR, (1-HR); mean +/- 95% CI over N_RUNS."])
    print("saved:", path)


(q_mean, q_ci), agg = benchmark()
write_csv(r"/Users/alejandrogomez-paz/Desktop/RAG/benchmark_results.csv", (q_mean, q_ci), agg)

saved: /Users/alejandrogomez-paz/Desktop/RAG/benchmark_results.csv
